# Phase 3 — Integration E2E Test (TASK-14)

Notebook này chạy 15 câu hỏi thử nghiệm qua toàn bộ pipeline RAG:

```
question → QueryPlanner → SubgraphExtractor → HybridSearch → ContextAssembler → AnswerGenerator
```

**Phạm vi [A]:** Đất đai (chuyển mục đích SDĐ + cấp sổ đỏ lần đầu), TP.HCM + Đồng Nai + Toàn quốc  
**DoD cần đạt:**
- DoD 1: Câu hỏi Đất đai TP.HCM trả lời trong < 30s
- DoD 2: ≥ 2 câu hỏi cho mỗi thủ tục
- DoD 3: Câu hỏi thiếu jurisdiction → `confirmation_needed=True`
- DoD 4: Negative test khai sinh TP.HCM vs Đồng Nai → không bịa sự khác biệt
- DoD 5: Ghi kết quả + nhận xét vào notebook

> **Lưu ý citation**: LLM có thể dùng format tắt `[Điều X, Luật Y]` thay vì format chuẩn `[Điều X, Văn bản Y]`.  
> `parse_citations()` chỉ bắt format chuẩn — citation count = 0 không có nghĩa LLM không trích dẫn.  
> Đánh giá chất lượng trích dẫn bằng mắt qua phần TRẢ LỜI.

In [1]:
import os
import sys
import json
import time
import logging
from pathlib import Path

# Tìm project root bất kể notebook được chạy từ đâu
_cwd = Path.cwd()
_project_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'CLAUDE.md').exists()),
    _cwd,
)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
print(f'Project root: {_project_root}')

from dotenv import load_dotenv
load_dotenv(_project_root / '.env')

logging.basicConfig(level=logging.WARNING)
print('Import OK')

Project root: /Users/daonguyentandat/Documents/University/2526_Sem2/Thesis/vn-legal-graphrag
Import OK


In [2]:
import anthropic
from neo4j import GraphDatabase
from qdrant_client import QdrantClient

from src.ingestion.vectorizer import load_model
from src.pipeline import run_pipeline

# Khởi tạo clients một lần, dùng lại cho tất cả câu hỏi
neo4j_driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD', '')),
)
qdrant_client = QdrantClient(
    host=os.getenv('QDRANT_HOST', 'localhost'),
    port=int(os.getenv('QDRANT_PORT', '6333')),
)
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
model = load_model()

print('Clients OK')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Clients OK


In [3]:
results = []  # lưu kết quả để đánh giá cuối

def ask(question: str, label: str = '') -> dict:
    """Chạy pipeline và in kết quả gọn."""
    print(f"\n{'='*70}")
    if label:
        print(f"[{label}]")
    print(f"CÂU HỎI: {question}")
    print('='*70)

    result = run_pipeline(
        question,
        neo4j_driver=neo4j_driver,
        qdrant_client=qdrant_client,
        anthropic_client=anthropic_client,
        model=model,
    )

    if result['confirmation_needed']:
        print('⚠️  CẦN XÁC NHẬN:')
        print(result['confirmation_prompt'])
    else:
        print(f"📊 LCCIDs: {result['lccids_count']}  |  Top-k: {result['top_k_count']}  |  Context: ~{result['context_tokens']} tokens")
        print(f"\n💬 TRẢ LỜI:\n{result['answer']}")
        if result['citations']:
            print(f"\n📌 CITATIONS parsed ({len(result['citations'])}): {result['citations']}")
        else:
            print('\n📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)')

    print(f"\n⏱️  {result['elapsed_seconds']}s")
    return result

print('ask() ready')

ask() ready


## 1. Chuyển mục đích sử dụng đất — TP.HCM (DoD 1 + DoD 2)

In [4]:
# DoD 1: câu hỏi chuẩn, phải trả lời trong < 30s
r = ask(
    'Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?',
    label='Q01 | CMĐSDĐ TP.HCM | DoD-1'
)
results.append(r)

# DoD 1 checks
assert r['elapsed_seconds'] < 30, f'TIMEOUT: {r["elapsed_seconds"]}s'
assert not r['confirmation_needed'], 'Câu hỏi có đủ jurisdiction — không được confirmation_needed'
# Citation count là soft check vì LLM có thể dùng format tắt
if len(r['citations']) >= 1:
    print('✅ DoD 1 PASS (có citation theo format chuẩn)')
else:
    print('⚠️  DoD 1 PARTIAL — dưới 30s, có trả lời, nhưng citation format chưa chuẩn (kiểm tra thủ công)')


[Q01 | CMĐSDĐ TP.HCM | DoD-1]
CÂU HỎI: Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?


📊 LCCIDs: 4  |  Top-k: 25  |  Context: ~2926 tokens

💬 TRẢ LỜI:
## Điều kiện chuyển mục đích sử dụng đất tại TP.HCM

Dựa trên các quy định pháp luật hiện hành, điều kiện chuyển mục đích sử dụng đất (áp dụng chung, bao gồm TP.HCM) như sau:

---

### 1. Các trường hợp phải xin phép cơ quan nhà nước có thẩm quyền

Người sử dụng đất phải được cơ quan có thẩm quyền cho phép khi chuyển sang các mục đích như:
- Chuyển đất phi nông nghiệp được giao không thu tiền sang loại phi nông nghiệp khác được giao có thu tiền hoặc cho thuê đất [Điều 121, Khoản 1, Điểm d, Văn bản luat-dat-dai-2024]
- Chuyển đất xây dựng công trình sự nghiệp, đất sử dụng vào mục đích công cộng có mục đích kinh doanh sang đất sản xuất, kinh doanh phi nông nghiệp [Điều 121, Khoản 1, Điểm e, Văn bản luat-dat-dai-2024]

---

### 2. Nghĩa vụ tài chính khi chuyển mục đích

Khi chuyển mục đích sử dụng đất thuộc diện phải xin phép, người sử dụng đất **phải thực hiện nghĩa vụ tài chính** theo quy định; chế độ sử dụng đất, quyền và 

In [5]:
r = ask(
    'Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?',
    label='Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình'
)
results.append(r)


[Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình]
CÂU HỎI: Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?


📊 LCCIDs: 4  |  Top-k: 25  |  Context: ~2962 tokens

💬 TRẢ LỜI:
## Hộ gia đình chuyển đất nông nghiệp sang đất ở tại TP.HCM

**Có thể được chuyển**, nhưng phải đáp ứng các điều kiện sau:

---

### 1. Phải xin phép cơ quan nhà nước có thẩm quyền

Việc chuyển đất nông nghiệp sang đất phi nông nghiệp (bao gồm đất ở) thuộc trường hợp **bắt buộc phải xin phép**.
[Điều 121, Khoản 1, Điểm b, Văn bản luat-dat-dai-2024]

---

### 2. Điều kiện về quy hoạch

Căn cứ để được phép chuyển mục đích sử dụng **đất nông nghiệp trong khu dân cư** hoặc **đất nông nghiệp trong cùng thửa đất có đất ở sang đất ở** đối với hộ gia đình, cá nhân là:
- Quy hoạch sử dụng đất **cấp huyện**, **hoặc**
- Quy hoạch chung, **hoặc**
- Quy hoạch phân khu theo pháp luật về quy hoạch đô thị và nông thôn

đã được cơ quan có thẩm quyền phê duyệt.

[Điều 116, Khoản 5, Văn bản luat-dat-dai-2024]

---

### 3. Thẩm quyền quyết định

- **UBND cấp huyện** (quận/huyện tại TP.HCM) có thẩm quyền cho phép chuyển mục đích sử dụng đất đố

In [6]:
r = ask(
    'Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?',
    label='Q03 | CMĐSDĐ TP.HCM — hồ sơ'
)
results.append(r)


[Q03 | CMĐSDĐ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?


📊 LCCIDs: 5  |  Top-k: 25  |  Context: ~2935 tokens

💬 TRẢ LỜI:
## Hồ sơ xin chuyển mục đích sử dụng đất

Dựa trên thông tin trong context được cung cấp, **Luật Đất đai 2024 chỉ quy định về trình tự, thủ tục chuyển mục đích sử dụng đất theo hướng chung**, cụ thể:

### Trình tự thực hiện:

1. **Người sử dụng đất nộp hồ sơ** đề nghị chuyển mục đích sử dụng đất theo quy định.
   [Điều 227, Khoản 1, Văn bản luat-dat-dai-2024]

2. **Cơ quan quản lý đất đai kiểm tra** các điều kiện chuyển mục đích sử dụng đất. Nếu hồ sơ chưa đảm bảo, người sử dụng đất sẽ được hướng dẫn bổ sung và nộp lại.
   [Điều 227, Khoản 2, Văn bản luat-dat-dai-2024]

---

### ⚠️ Lưu ý quan trọng:

**Context được cung cấp không có quy định chi tiết về thành phần hồ sơ cụ thể** (danh sách các giấy tờ cần nộp) khi xin chuyển mục đích sử dụng đất tại TP.HCM.

Để có thông tin chính xác về thành phần hồ sơ, bạn cần tham khảo thêm:
- **Nghị định hướng dẫn thi hành Luật Đất đai 2024** của Chính phủ (quy định chi tiết thành phần

In [7]:
r = ask(
    'Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?',
    label='Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính'
)
results.append(r)


[Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính]
CÂU HỎI: Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?


📊 LCCIDs: 5  |  Top-k: 25  |  Context: ~2995 tokens

💬 TRẢ LỜI:
## Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở

### 1. Nguyên tắc chung về nghĩa vụ tài chính

Khi chuyển mục đích sử dụng đất sang đất ở (thuộc các trường hợp phải xin phép), người sử dụng đất **phải thực hiện nghĩa vụ tài chính** theo quy định của pháp luật. [Điều 121, Khoản 2, Văn bản luat-dat-dai-2024]

Cụ thể, các trường hợp phải xin phép và nộp tiền sử dụng đất bao gồm:
- Chuyển đất phi nông nghiệp không phải là đất ở **sang đất ở** [Điều 121, Khoản 1, Điểm đ, Văn bản luat-dat-dai-2024]

### 2. Căn cứ tính tiền sử dụng đất

Tiền sử dụng đất khi chuyển mục đích sử dụng đất của hộ gia đình, cá nhân được tính theo **bảng giá đất**. [Điều 159, Khoản 1, Điểm a, Văn bản luat-dat-dai-2024]

> **Lưu ý quan trọng:** Context không cung cấp mức giá cụ thể tại TP.HCM. Bảng giá đất do UBND cấp tỉnh (tại TP.HCM là UBND TP.HCM) ban hành và có thể thay đổi theo từng thời kỳ.

### 3. Các trường hợp được miễn/không p

## 2. Chuyển mục đích sử dụng đất — Đồng Nai

In [8]:
r = ask(
    'Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?',
    label='Q05 | CMĐSDĐ Đồng Nai — quy trình'
)
results.append(r)


[Q05 | CMĐSDĐ Đồng Nai — quy trình]
CÂU HỎI: Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?


📊 LCCIDs: 4  |  Top-k: 25  |  Context: ~3016 tokens

💬 TRẢ LỜI:
# Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai

## 1. Các trường hợp phải xin phép

Việc chuyển đất nông nghiệp sang đất ở **bắt buộc phải xin phép** cơ quan nhà nước có thẩm quyền, cụ thể:

- Chuyển đất nông nghiệp sang đất phi nông nghiệp (bao gồm đất ở) [Điều 121, Khoản 1, Điểm b, Văn bản luat-dat-dai-2024]
- Chuyển đất phi nông nghiệp không phải là đất ở sang đất ở [Điều 121, Khoản 1, Điểm đ, Văn bản luat-dat-dai-2024]

## 2. Căn cứ để được phép chuyển mục đích

Đối với **hộ gia đình, cá nhân**, căn cứ để được phép chuyển đất nông nghiệp sang đất ở là phải phù hợp với một trong các loại quy hoạch sau đã được cơ quan có thẩm quyền phê duyệt:
- Quy hoạch sử dụng đất cấp huyện, **hoặc**
- Quy hoạch chung, **hoặc**
- Quy hoạch phân khu theo quy định của pháp luật về quy hoạch đô thị và nông thôn

[Điều 116, Khoản 5, Văn bản luat-dat-dai-2024]

## 3. Điều kiện đặc biệt với đất trồng lúa

Nếu diệ

In [9]:
r = ask(
    'Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?',
    label='Q06 | CMĐSDĐ Đồng Nai — thời hạn'
)
results.append(r)


[Q06 | CMĐSDĐ Đồng Nai — thời hạn]
CÂU HỎI: Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?


📊 LCCIDs: 5  |  Top-k: 25  |  Context: ~3002 tokens

💬 TRẢ LỜI:
## Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai

Dựa trên thông tin trong context được cung cấp, **Luật Đất đai 2024 chỉ quy định trình tự, thủ tục cho phép chuyển mục đích sử dụng đất** theo các bước:

1. Người sử dụng đất nộp hồ sơ đề nghị chuyển mục đích sử dụng đất [Điều 227, Khoản 1, Văn bản luat-dat-dai-2024]

2. Cơ quan có chức năng quản lý đất đai kiểm tra các điều kiện chuyển mục đích sử dụng đất; trường hợp hồ sơ chưa bảo đảm thì hướng dẫn bổ sung [Điều 227, Khoản 2, Văn bản luat-dat-dai-2024]

3. Lập hồ sơ trình Ủy ban nhân dân cấp có thẩm quyền ban hành quyết định cho phép chuyển mục đích sử dụng đất [Điều 227, Khoản 3, Điểm a, Văn bản luat-dat-dai-2024]

---

⚠️ **Lưu ý quan trọng:** Context được cung cấp **không có thông tin về thời hạn cụ thể** (số ngày) để giải quyết hồ sơ chuyển mục đích sử dụng đất, cũng như **không có quy định riêng của tỉnh Đồng Nai** về vấn đề này.

Để biết thời h

## 3. Cấp sổ đỏ lần đầu — TP.HCM (DoD 2)

In [10]:
r = ask(
    'Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?',
    label='Q07 | Cấp sổ đỏ TP.HCM — điều kiện'
)
results.append(r)


[Q07 | Cấp sổ đỏ TP.HCM — điều kiện]
CÂU HỎI: Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?


📊 LCCIDs: 5  |  Top-k: 25  |  Context: ~3010 tokens

💬 TRẢ LỜI:
## Điều kiện cấp Giấy chứng nhận quyền sử dụng đất lần đầu

Dựa trên thông tin trong context, các điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất (sổ đỏ) lần đầu được quy định như sau:

---

### 1. Nguyên tắc chung

- GCN được cấp theo từng thửa đất cho người **có nhu cầu và đủ điều kiện** theo quy định pháp luật. [Điều 135, Khoản 1, Văn bản luat-dat-dai-2024]
- Người sử dụng đất phải **hoàn thành nghĩa vụ tài chính** (nộp tiền sử dụng đất, thuế...) trước khi được cấp GCN. Trừ trường hợp được miễn hoặc ghi nợ. [Điều 135, Khoản 3, Văn bản luat-dat-dai-2024]

---

### 2. Trường hợp có giấy tờ về quyền sử dụng đất

Hộ gia đình, cá nhân đang **sử dụng đất ổn định** và có một trong các loại giấy tờ được lập **trước ngày 15/10/1993** thì được cấp GCN và **không phải nộp tiền sử dụng đất**. [Điều 137, Khoản 1, Văn bản luat-dat-dai-2024]

Ngoài ra, nếu có giấy tờ của **nông, lâm trường quốc doanh** giao đất làm nhà ở trước

In [11]:
r = ask(
    'Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?',
    label='Q08 | Cấp sổ đỏ TP.HCM — hồ sơ'
)
results.append(r)


[Q08 | Cấp sổ đỏ TP.HCM — hồ sơ]
CÂU HỎI: Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?


📊 LCCIDs: 4  |  Top-k: 25  |  Context: ~2798 tokens

💬 TRẢ LỜI:
## Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu

Dựa trên thông tin trong context được cung cấp, **context không có quy định cụ thể về thành phần hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu** tại TP.HCM hay trên phạm vi cả nước.

---

### Những gì context có đề cập liên quan:

**1. Nguyên tắc cấp Giấy chứng nhận:**
- Cấp theo từng thửa đất cho người có quyền sử dụng đất có nhu cầu và đủ điều kiện [Điều 135, Khoản 1, Văn bản luat-dat-dai-2024]

**2. Các giấy tờ về quyền sử dụng đất được chấp nhận** (lập trước ngày 15/10/1993) để được cấp Giấy chứng nhận không phải nộp tiền sử dụng đất, bao gồm:
- Giấy tờ do cơ quan có thẩm quyền thuộc chế độ cũ cấp (Bằng khoán điền thổ, văn tự đoạn mãi...) [Điều 137, Khoản 1, Điểm b, Văn bản luat-dat-dai-2024]
- Giấy chứng nhận quyền sử dụng đất tạm thời hoặc có tên trong Sổ đăng ký ruộng đất, Sổ địa chính [Điều 137, Khoản 1, Điểm c, Văn bản luat-dat-dai-2024]

In [12]:
r = ask(
    'Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?',
    label='Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ'
)
results.append(r)


[Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ]
CÂU HỎI: Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?


📊 LCCIDs: 4  |  Top-k: 25  |  Context: ~2959 tokens

💬 TRẢ LỜI:
## Cấp sổ đỏ lần đầu cho đất không có giấy tờ tại TP.HCM

**Có thể được cấp sổ đỏ lần đầu**, theo quy định tại Điều 138 Luật Đất đai 2024, áp dụng cho hộ gia đình, cá nhân đang sử dụng đất ổn định mà **không có giấy tờ về quyền sử dụng đất**, không vi phạm pháp luật đất đai, không thuộc trường hợp đất giao không đúng thẩm quyền. [Điều 138, Văn bản luat-dat-dai-2024]

---

### Điều kiện cụ thể theo từng thời kỳ sử dụng đất:

#### 1. Sử dụng đất **trước ngày 18/12/1980**
- Được cấp sổ đỏ nếu **được UBND cấp xã nơi có đất xác nhận không có tranh chấp**.
[Điều 138, Khoản 1, Văn bản luat-dat-dai-2024]

#### 2. Đối với phần diện tích đất còn lại (xác định theo hiện trạng):
- Nếu hiện trạng là **đất phi nông nghiệp không phải đất ở**: được công nhận theo quy định tương ứng.
- Nếu hiện trạng là **đất nông nghiệp**: được công nhận theo hình thức Nhà nước giao đất **không thu tiền sử dụng đất**; nếu muốn chuyển sang đất phi nông ngh

## 4. Cấp sổ đỏ lần đầu — Đồng Nai

In [13]:
r = ask(
    'Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?',
    label='Q10 | Cấp sổ đỏ Đồng Nai — cơ quan'
)
results.append(r)


[Q10 | Cấp sổ đỏ Đồng Nai — cơ quan]
CÂU HỎI: Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?


📊 LCCIDs: 5  |  Top-k: 25  |  Context: ~3010 tokens

💬 TRẢ LỜI:
## Cơ quan tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu

Dựa trên thông tin trong context được cung cấp, có thể xác định một số thông tin liên quan như sau:

**Về thẩm quyền cấp Giấy chứng nhận:**

- **Ủy ban nhân dân cấp tỉnh** có thẩm quyền cấp Giấy chứng nhận cho người sử dụng đất là tổ chức trong nước, tổ chức tôn giáo, tổ chức nước ngoài, tổ chức kinh tế có vốn đầu tư nước ngoài, v.v., và có thể **ủy quyền cho cơ quan có chức năng quản lý đất đai cùng cấp** thực hiện. [Điều 136, Khoản 1, Điểm a, Văn bản luat-dat-dai-2024]

- **Tổ chức đăng ký đất đai** thực hiện đăng ký cho các đối tượng là tổ chức trong nước, tổ chức tôn giáo, tổ chức nước ngoài có chức năng ngoại giao, tổ chức kinh tế có vốn đầu tư nước ngoài, v.v. [Điều 136, Khoản 2, Điểm a, Văn bản luat-dat-dai-2024]

- **Tổ chức đăng ký đất đai và chi nhánh** được sử dụng con dấu của mình để thực hiện cấp Giấy chứng nhận. [Điều 136, Khoản

In [14]:
r = ask(
    'Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?',
    label='Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí'
)
results.append(r)


[Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí]
CÂU HỎI: Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?


📊 LCCIDs: 5  |  Top-k: 25  |  Context: ~2836 tokens

💬 TRẢ LỜI:
## Phí thẩm định hồ sơ cấp sổ đỏ lần đầu tại Đồng Nai

Theo Nghị quyết số 22/2024/NQ-HĐND của Hội đồng nhân dân tỉnh Đồng Nai, mức phí thẩm định hồ sơ cấp Giấy chứng nhận (sổ đỏ) **lần đầu** như sau:

### Đối với Hộ gia đình, cá nhân

| Loại hồ sơ | Nộp trực tiếp | Nộp trực tuyến |
|---|:---:|:---:|
| Cấp GCN về quyền sử dụng đất | 880.000 đồng | 836.000 đồng |
| Cấp GCN về tài sản | 980.000 đồng | 931.000 đồng |
| Cấp GCN cả đất và tài sản gắn liền với đất | 1.250.000 đồng | 1.187.500 đồng |

[Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai]

### Đối với Tổ chức

| Loại hồ sơ | Nộp trực tiếp | Nộp trực tuyến |
|---|:---:|:---:|
| Cấp GCN về quyền sử dụng đất | 1.260.000 đồng | 1.197.000 đồng |
| Cấp GCN về tài sản | 1.840.000 đồng | 1.748.000 đồng |
| Cấp GCN cả đất và tài sản gắn liền với đất | 2.090.000 đồng | 1.985.500 đồng |

[Phụ lục I, Văn bản nghi-quyet-22-2024-nq-hdnd-dong-nai]

### Lưu ý quan trọng

- **Đồ

## 5. DoD 3 — Thiếu jurisdiction → confirmation_needed

In [15]:
r = ask(
    'Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?',
    label='Q12 | Thiếu jurisdiction | DoD-3'
)
results.append(r)
assert r['confirmation_needed'] is True, 'Phải confirmation_needed=True khi thiếu jurisdiction'
assert r['confirmation_prompt'] is not None
print('✅ DoD 3 PASS')


[Q12 | Thiếu jurisdiction | DoD-3]
CÂU HỎI: Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?
⚠️  CẦN XÁC NHẬN:
Bất động sản / đất đai của bạn thuộc tỉnh/thành phố nào? (TP. Hồ Chí Minh / Đồng Nai / địa phương khác)

⏱️  1.1s
✅ DoD 3 PASS


## 6. DoD 4 — Negative test: khai sinh TP.HCM vs Đồng Nai

In [16]:
r = ask(
    'Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?',
    label='Q13 | Negative — khai sinh địa phương | DoD-4'
)
results.append(r)
# Khai sinh là thủ tục toàn quốc → pipeline trả về empty context (chưa có data Hộ tịch)
# hoặc nói rõ không có sự khác biệt. Không được bịa sự khác biệt.
print('\n⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại')


[Q13 | Negative — khai sinh địa phương | DoD-4]
CÂU HỎI: Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?


📊 LCCIDs: 0  |  Top-k: 0  |  Context: ~0 tokens

💬 TRẢ LỜI:
Không tìm thấy văn bản pháp luật liên quan đến câu hỏi này.

📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)

⏱️  1.27s

⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại


## 7. Gap 3 — câu hỏi cần traversal [:IMPLEMENTS]

In [17]:
r = ask(
    'Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?',
    label='Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM'
)
results.append(r)


[Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM]
CÂU HỎI: Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?


📊 LCCIDs: 4  |  Top-k: 25  |  Context: ~2992 tokens

💬 TRẢ LỜI:
## Quy định về chuyển mục đích sử dụng đất theo Luật Đất đai 2024 và Nghị định 102/2024/NĐ-CP

Dựa trên thông tin trong context được cung cấp, tôi **không có đủ thông tin** để trả lời cụ thể về các quy định riêng áp dụng tại **TP.HCM** trong Nghị định 102/2024/NĐ-CP.

Tuy nhiên, context cung cấp một số quy định chung liên quan đến chuyển mục đích sử dụng đất như sau:

---

### 1. Căn cứ chuyển mục đích sử dụng đất

Đối với hộ gia đình, cá nhân muốn chuyển đất nông nghiệp trong khu dân cư hoặc đất phi nông nghiệp sang đất ở, căn cứ là **quy hoạch sử dụng đất cấp huyện** hoặc **quy hoạch chung/quy hoạch phân khu** đã được cơ quan có thẩm quyền phê duyệt.
[Điều 116, Khoản 5, Văn bản luat-dat-dai-2024]

### 2. Điều kiện chuyển mục đích sử dụng đất

- Đối với **đất trồng lúa, đất rừng đặc dụng, đất rừng phòng hộ, đất rừng sản xuất**: phải có **Nghị quyết của HĐND cấp tỉnh** và đáp ứng căn cứ tại Điều 116 Luật Đất đai.
[Điều 122

In [18]:
r = ask(
    'Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?',
    label='Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM'
)
results.append(r)


[Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM]
CÂU HỎI: Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?


📊 LCCIDs: 1  |  Top-k: 25  |  Context: ~2992 tokens

💬 TRẢ LỜI:
## Bảng giá đất và ảnh hưởng đến tiền sử dụng đất khi chuyển mục đích sử dụng đất

### 1. Bảng giá đất được áp dụng khi nào?

Bảng giá đất được sử dụng để **tính tiền sử dụng đất khi chuyển mục đích sử dụng đất của hộ gia đình, cá nhân** [Điều 159, Khoản 1, Điểm a, Văn bản luat-dat-dai-2024].

---

### 2. Thời điểm áp dụng bảng giá đất TP.HCM năm 2025

Căn cứ theo quy định, **bảng giá đất lần đầu** sẽ được xây dựng và áp dụng từ ngày **01/01/2026**, không phải năm 2025. Cụ thể:

> "Ủy ban nhân dân cấp tỉnh xây dựng, trình Hội đồng nhân dân cùng cấp quyết định bảng giá đất lần đầu để công bố và áp dụng **từ ngày 01 tháng 01 năm 2026**."

[Điều 159, Khoản 3, Văn bản luat-dat-dai-2024]

---

### 3. Thời điểm tính tiền sử dụng đất khi chuyển mục đích

Thời điểm để xác định giá đất tính tiền sử dụng đất là **thời điểm Nhà nước ban hành quyết định cho phép chuyển mục đích sử dụng đất** [Điều 155, Khoản 3, Điểm a, Văn bản luat-da

## 8. Tổng kết kết quả

In [19]:
print('\n' + '='*70)
print('TỔNG KẾT PIPELINE E2E TEST — TASK-14')
print('='*70)

total = len(results)
confirmed = sum(1 for r in results if r['confirmation_needed'])
answered = total - confirmed
with_parsed_citations = sum(1 for r in results if not r['confirmation_needed'] and r['citations'])
avg_elapsed = sum(r['elapsed_seconds'] for r in results) / total if total else 0

print(f'  Tổng câu hỏi:             {total}')
print(f'  Câu trả lời được:         {answered}')
print(f'  Cần xác nhận jurisdiction: {confirmed}')
print(f'  Có citation (format chuẩn): {with_parsed_citations}/{answered}')
print(f'  Thời gian TB:             {avg_elapsed:.1f}s')
if total:
    print(f'  Max elapsed:              {max(r["elapsed_seconds"] for r in results):.1f}s')

print('\nChi tiết:')
for i, r in enumerate(results, 1):
    if r['confirmation_needed']:
        status = '⚠️  CONFIRM'
    elif r['citations']:
        status = f'✅ {len(r["citations"])} cite'
    else:
        status = '⚡ 0 cite*'
    print(f'  Q{i:02d}: {status:12} {r["elapsed_seconds"]:5.1f}s | LCCIDs={r["lccids_count"]:4d} | {r["question"][:55]}')

print('\n* 0 cite = LLM có thể dùng format tắt, kiểm tra thủ công')


TỔNG KẾT PIPELINE E2E TEST — TASK-14
  Tổng câu hỏi:             15
  Câu trả lời được:         14
  Cần xác nhận jurisdiction: 1
  Có citation (format chuẩn): 12/14
  Thời gian TB:             15.1s
  Max elapsed:              23.5s

Chi tiết:
  Q01: ✅ 3 cite      23.5s | LCCIDs=   4 | Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là 
  Q02: ✅ 2 cite      18.0s | LCCIDs=   4 | Hộ gia đình có được chuyển đất nông nghiệp sang đất ở t
  Q03: ✅ 2 cite      13.0s | LCCIDs=   5 | Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm nh
  Q04: ✅ 2 cite      15.7s | LCCIDs=   5 | Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang
  Q05: ✅ 3 cite      22.0s | LCCIDs=   4 | Quy trình chuyển mục đích sử dụng đất nông nghiệp sang 
  Q06: ✅ 2 cite      10.2s | LCCIDs=   5 | Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất t
  Q07: ✅ 10 cite     19.6s | LCCIDs=   5 | Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất
  Q08: ✅ 1 cite      18.2s | LCCIDs=   4 | Hồ sơ đăng ký cấp 

## 9. Nhận xét & Vấn đề cần theo dõi

### ✅ Điều tốt
- Tất cả DoD cơ học đều pass: thời gian < 30s, DoD 3 confirmation_needed, DoD 4 không bịa thông tin.
- **Q11** (phí sổ đỏ Đồng Nai, 118 LCCIDs): trả lời xuất sắc — bảng phí đầy đủ, trích dẫn rõ ràng. Khi subgraph hẹp và đúng, pipeline hoạt động rất tốt.
- **Q13** (khai sinh — không có data Hộ tịch): trả về "Không tìm thấy văn bản" thay vì bịa thông tin.

### ⚠️ Vấn đề 1: LCCID explosion (>2000 cho nhiều câu hỏi)
**Root cause:** Stage 1 trả về Luật Đất đai 2024 → Stage 2 traversal  kéo toàn bộ 17 văn bản → ~3000 LCCIDs.  
**Biểu hiện:** Hybrid search chọn top-10 nhưng không trúng nội dung → LLM nói "không đủ thông tin".  
**Hướng xử lý cho Phase 4:** Giảm  Stage 1 xuống 2-3, hoặc thêm Stage 1.5 rerank theo procedure keyword.

### ⚠️ Vấn đề 2: Citation format (đã fix trong build_prompt)
LLM tự nhiên dùng  thay vì  mà regex yêu cầu.  
**Đã sửa** prompt để bắt buộc từ khoá "Văn bản" với ví dụ cụ thể. Cần re-run notebook sau khi push fix.

### 📋 Kết luận Gate Phase 3
Pipeline đã hoạt động end-to-end. Chất lượng retrieval với câu hỏi hẹp (Q11) rất tốt.  
Vấn đề LCCID explosion là **retrieval quality issue** sẽ được cải thiện qua tuning — không phải lỗi logic pipeline.  
Sẵn sàng bắt đầu Phase 4 với cơ sở dữ liệu Đất đai hiện có.


In [20]:
# Đóng clients
neo4j_driver.close()
print('Clients closed.')

Clients closed.
